In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : data_quality_checks
# PURPOSE    : Reusable data quality validation functions
#              Run after each layer to ensure data integrity
# USAGE      : %run ../utils/data_quality_checks
#              Then call functions in any notebook
# CHECKS:
#   - Row count validation
#   - Null count per column
#   - Duplicate check
#   - Schema validation
#   - Value range validation
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
# ============================================================
# FUNCTION 1 - ROW COUNT CHECK
# Validates that a table has expected number of rows
# Used to confirm no data was lost between layers
# ============================================================

def check_row_count(table_name, expected_count=None):
    """
    Checks row count of a table.
    If expected_count provided, validates against it.

    Usage:
    check_row_count("fincompliance_catalog.bronze.customers")
    check_row_count("fincompliance_catalog.silver.orders",
                    expected_count=99441)
    """
    actual_count = spark.table(table_name).count()

    if expected_count:
        if actual_count == expected_count:
            print(f"PASS : {table_name}")
            print(f"Row count = {actual_count:,} "
                  f"(matches expected {expected_count:,})")
        else:
            print(f"FAIL : {table_name}")
            print(f"Expected {expected_count:,} "
                  f"but got {actual_count:,}")
    else:
        print(f"INFO : {table_name}")
        print(f"Row count = {actual_count:,}")

    return actual_count

print("check_row_count function defined")

In [0]:
# ============================================================
# FUNCTION 2 - NULL CHECK
# Checks for null values in specified columns
# Fails if null count exceeds threshold
# ============================================================

def check_nulls(table_name, columns, threshold=0):
    """
    Checks for null values in specified columns.
    threshold = maximum allowed nulls (default 0)

    Usage:
    check_nulls(
        "fincompliance_catalog.silver.customers",
        ["customer_id", "customer_state"],
        threshold=0
    )
    """
    from pyspark.sql.functions import col, sum as spark_sum, when

    df = spark.table(table_name)
    print(f"\nNull check : {table_name}")

    all_passed = True
    for c in columns:
        null_count = df \
            .filter(col(c).isNull()) \
            .count()

        if null_count <= threshold:
            print(f"  PASS : {c:<35} "
                  f": {null_count:,} nulls")
        else:
            print(f"  FAIL : {c:<35} "
                  f": {null_count:,} nulls "
                  f"(threshold={threshold})")
            all_passed = False

    if all_passed:
        print(f"  All null checks passed")
    else:
        print(f"  Some null checks FAILED")

    return all_passed

print("check_nulls function defined")

In [0]:
# ============================================================
# FUNCTION 3 - DUPLICATE CHECK
# Checks for duplicate values in primary key columns
# ============================================================

def check_duplicates(table_name, key_columns):
    """
    Checks for duplicate rows based on key columns.

    Usage:
    check_duplicates(
        "fincompliance_catalog.silver.orders",
        ["order_id"]
    )
    check_duplicates(
        "fincompliance_catalog.silver.order_items",
        ["order_id", "order_item_id"]
    )
    """
    df = spark.table(table_name)
    total_rows = df.count()
    unique_rows = df.dropDuplicates(key_columns).count()
    duplicate_count = total_rows - unique_rows

    print(f"\nDuplicate check : {table_name}")
    print(f"  Key columns : {key_columns}")
    print(f"  Total rows  : {total_rows:,}")
    print(f"  Unique rows : {unique_rows:,}")

    if duplicate_count == 0:
        print(f"  PASS : No duplicates found")
    else:
        print(f"  FAIL : {duplicate_count:,} duplicates found")

    return duplicate_count

print("check_duplicates function defined")

In [0]:
# ============================================================
# FUNCTION 4 - SCHEMA CHECK
# Validates that expected columns exist in table
# ============================================================

def check_schema(table_name, expected_columns):
    """
    Checks that all expected columns exist in table.

    Usage:
    check_schema(
        "fincompliance_catalog.silver.orders",
        ["order_id", "delivery_status",
         "delivery_delay_days", "order_year"]
    )
    """
    df = spark.table(table_name)
    actual_columns = df.columns

    print(f"\nSchema check : {table_name}")
    all_passed = True

    for c in expected_columns:
        if c in actual_columns:
            print(f"  PASS : {c:<35} exists")
        else:
            print(f"  FAIL : {c:<35} MISSING")
            all_passed = False

    if all_passed:
        print(f"  All schema checks passed")
    else:
        print(f"  Some schema checks FAILED")

    return all_passed

print("check_schema function defined")

In [0]:
# ============================================================
# FUNCTION 5 - VALUE RANGE CHECK
# Validates that numeric columns are within expected range
# ============================================================

def check_value_range(table_name, column,
                      min_val=None, max_val=None):
    """
    Checks that values in a column are within expected range.

    Usage:
    check_value_range(
        "fincompliance_catalog.silver.order_reviews",
        "review_score",
        min_val=1,
        max_val=5
    )
    """
    from pyspark.sql.functions import col, min as spark_min
    from pyspark.sql.functions import max as spark_max

    df = spark.table(table_name)
    stats = df.filter(col(column).isNotNull()) \
              .agg(
                  spark_min(column).alias("min_val"),
                  spark_max(column).alias("max_val")
              ).collect()[0]

    actual_min = stats["min_val"]
    actual_max = stats["max_val"]

    print(f"\nValue range check : {table_name}.{column}")
    print(f"  Actual range : {actual_min} to {actual_max}")

    passed = True
    if min_val and actual_min < min_val:
        print(f"  FAIL : min {actual_min} "
              f"below expected {min_val}")
        passed = False
    if max_val and actual_max > max_val:
        print(f"  FAIL : max {actual_max} "
              f"above expected {max_val}")
        passed = False
    if passed:
        print(f"  PASS : values within expected range")

    return passed

print("check_value_range function defined")

In [0]:
# ============================================================
# RUN ALL DATA QUALITY CHECKS
# Validates all bronze, silver and gold tables
# ============================================================

print("=" * 60)
print("DATA QUALITY CHECKS — ALL LAYERS")
print("=" * 60)

# ────────────────────────────────────────────────────────────
# BRONZE LAYER CHECKS
# ────────────────────────────────────────────────────────────
print("\nBRONZE LAYER:")
bronze_tables = {
    "customers"           : 99441,
    "geolocation"         : 1000163,
    "orders"              : 99441,
    "order_items"         : 112650,
    "order_payments"      : 103886,
    "order_reviews"       : 104162,
    "products"            : 32951,
    "category_translation": 71,
    "sellers"             : 3095
}
for table, expected in bronze_tables.items():
    check_row_count(
        f"{BRONZE_DB}.{table}",
        expected_count=expected
    )

# ────────────────────────────────────────────────────────────
# SILVER LAYER CHECKS
# ────────────────────────────────────────────────────────────
print("\nSILVER LAYER:")

# Row counts
silver_tables = {
    "customers"      : 99441,
    "orders"         : 99441,
    "order_items"    : 112650,
    "order_payments" : 103886,
    "sellers"        : 3095,
    "products"       : 32951,
    "geolocation"    : 19015
}
for table, expected in silver_tables.items():
    check_row_count(
        f"{SILVER_DB}.{table}",
        expected_count=expected
    )

# Null checks on key columns
check_nulls(
    f"{SILVER_DB}.customers",
    ["customer_id", "customer_state",
     "customer_region"]
)

check_nulls(
    f"{SILVER_DB}.orders",
    ["order_id", "order_status",
     "delivery_status", "order_year"]
)

check_nulls(
    f"{SILVER_DB}.sellers",
    ["seller_id", "seller_state", "seller_region"]
)

# Duplicate checks
check_duplicates(
    f"{SILVER_DB}.customers",
    ["customer_id"]
)

check_duplicates(
    f"{SILVER_DB}.orders",
    ["order_id"]
)

check_duplicates(
    f"{SILVER_DB}.order_items",
    ["order_id", "order_item_id"]
)

# Schema checks
check_schema(
    f"{SILVER_DB}.orders",
    ["order_year", "order_month",
     "delivery_status", "delivery_delay_days",
     "order_processing_days", "is_approved",
     "is_shipped", "silver_updated_at"]
)

check_schema(
    f"{SILVER_DB}.products",
    ["product_category_name_english",
     "product_size", "product_weight_class",
     "product_volume_cm3", "silver_updated_at"]
)

# Value range checks
check_value_range(
    f"{SILVER_DB}.order_reviews",
    "review_score",
    min_val=1,
    max_val=5
)

# ────────────────────────────────────────────────────────────
# GOLD LAYER CHECKS
# ────────────────────────────────────────────────────────────
print("\nGOLD LAYER:")

gold_tables = {
    "sales_summary"       : 23,
    "product_performance" : 72,
    "seller_performance"  : 2970,
    "customer_segments"   : 27,
    "delivery_analysis"   : 75,
    "payment_analysis"    : 12,
    "review_analysis"     : 73,
    "rfm_segments"        : 96478
}
for table, expected in gold_tables.items():
    check_row_count(
        f"{GOLD_DB}.{table}",
        expected_count=expected
    )

check_schema(
    f"{GOLD_DB}.rfm_segments",
    ["customer_segment", "rfm_score",
     "r_score", "f_score", "m_score",
     "recency_days", "frequency", "monetary"]
)

print("\n" + "=" * 60)
print("DATA QUALITY CHECKS COMPLETE")
print("=" * 60)